# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

1000 images processed
2000 images processed
3000 images processed
4000 images processed
5000 images processed
6000 images processed
7000 images processed
8000 images processed
9000 images processed
10000 images processed
11000 images processed
12000 images processed


In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [8]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.classifier = torch.nn.Sequential(torch.nn.Linear(2049, 256),
                                              torch.nn.Linear(256, 23))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.classifier(f)
        return y

In [9]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [10]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 530711


In [11]:
# Hyperparameters
epsilon = 1E-4

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=1000, checkpoint_path='resnet_transfer_double_FULL.pth', graph_save_directory='./fully_trained_double')

Epoch [0/1000], Train Loss: 2.55662, Val Loss: 2.97353 *, Train Acc: 0.24262, Validation Acc: 0.05667
Epoch [1/1000], Train Loss: 2.28735, Val Loss: 2.83886 *, Train Acc: 0.30917, Validation Acc: 0.08111
Epoch [2/1000], Train Loss: 2.14419, Val Loss: 2.74544 *, Train Acc: 0.34060, Validation Acc: 0.11944
Epoch [3/1000], Train Loss: 2.04087, Val Loss: 2.72282 *, Train Acc: 0.35726, Validation Acc: 0.08444
Epoch [4/1000], Train Loss: 2.02200, Val Loss: 2.65610 *, Train Acc: 0.35417, Validation Acc: 0.09889
Epoch [5/1000], Train Loss: 1.96608, Val Loss: 2.66612, Train Acc: 0.36762, Validation Acc: 0.10111
Epoch [6/1000], Train Loss: 1.92110, Val Loss: 2.65154 *, Train Acc: 0.38238, Validation Acc: 0.09500
Epoch [7/1000], Train Loss: 1.89524, Val Loss: 2.68737, Train Acc: 0.38690, Validation Acc: 0.08611
Epoch [8/1000], Train Loss: 1.87728, Val Loss: 2.66176, Train Acc: 0.38631, Validation Acc: 0.08778
Epoch [9/1000], Train Loss: 1.85001, Val Loss: 2.62435 *, Train Acc: 0.39631, Validation

In [7]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_model_FULL2.pth', criterion, test_loader, device)

Loss: 2.6393174817827014
Accuracy: 0.105
Off-by-One Accuracy: 0.32


In [8]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_model_FULL2.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.classifier.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device, num_epochs=100, checkpoint_path='resnet_finetuned_FULL2.pth', graph_save_directory='./fully_trained_finetuned2')

Epoch [0/100], Train Loss: 1.68687, Val Loss: 2.61890 *, Train Acc: 0.47333, Validation Acc: 0.09278
Epoch [1/100], Train Loss: 1.62408, Val Loss: 2.59066 *, Train Acc: 0.48357, Validation Acc: 0.09833
Epoch [2/100], Train Loss: 1.57716, Val Loss: 2.58182 *, Train Acc: 0.50976, Validation Acc: 0.09833
Epoch [3/100], Train Loss: 1.50285, Val Loss: 2.55751 *, Train Acc: 0.53631, Validation Acc: 0.10556
Epoch [4/100], Train Loss: 1.49657, Val Loss: 2.54775 *, Train Acc: 0.54083, Validation Acc: 0.10500
Epoch [5/100], Train Loss: 1.43639, Val Loss: 2.53144 *, Train Acc: 0.56381, Validation Acc: 0.10667
Epoch [6/100], Train Loss: 1.40819, Val Loss: 2.52803 *, Train Acc: 0.56952, Validation Acc: 0.10333
Epoch [7/100], Train Loss: 1.38983, Val Loss: 2.52551 *, Train Acc: 0.58310, Validation Acc: 0.10722
Epoch [8/100], Train Loss: 1.34968, Val Loss: 2.51719 *, Train Acc: 0.59512, Validation Acc: 0.10389
Epoch [9/100], Train Loss: 1.31362, Val Loss: 2.51500 *, Train Acc: 0.61512, Validation Acc

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-a614e8ff-8ed9-406e-87f8-2bbaf21e7f7f.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [9]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_finetuned_FULL2.pth', criterion, test_loader, device)

Loss: 2.5154944366878933
Accuracy: 0.11388888888888889
Off-by-One Accuracy: 0.4022222222222222
